# MuSeg-AI Thigh Segmentation — Lambda Cloud

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (nnU-Net `thigh-model3`)
on fat-fraction stacks using a GPU instance on [Lambda Cloud](https://cloud.lambdalabs.com).

## One-time local setup: extract weights from Docker

Run this on your local machine (Docker must be running):

```bash
docker pull fabianbalsiger/museg:thigh-model3
docker create --name museg_extract fabianbalsiger/museg:thigh-model3
docker cp museg_extract:/nnUNet_trained_models ./nnUNet_trained_models
docker rm museg_extract
```

## Upload data to the Lambda instance

From your local machine (use Git Bash / WSL on Windows):

```bash
# Weights (~1-2 GB)
rsync -avz ./eval_notebooks/nnUNet_trained_models ubuntu@<LAMBDA-IP>:~/

# NIfTI stacks
rsync -avz ./eval_notebooks/myosegmenTUM ubuntu@<LAMBDA-IP>:~/
```

## Steps in this notebook

1. Launch a **1× A10** instance on Lambda Cloud (~$1.29/hr)
2. Open **JupyterLab** from the instance dashboard
3. Upload this notebook via the JupyterLab upload button
4. Run all cells top to bottom
5. Download results: `rsync -avz ubuntu@<LAMBDA-IP>:~/museg_thigh_segs/ ./eval_notebooks/museg_thigh_segs/`
6. **Terminate the instance** when done

In [2]:
# Install dependencies (run once per instance)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "git+https://github.com/fabianbalsiger/museg-ai.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nnunet==1.7.1"])

# Distutils shim — needed on Python 3.12, harmless on 3.10
try:
    import distutils  # noqa: F401
except ModuleNotFoundError:
    try:
        import setuptools._distutils as _dt
        sys.modules['distutils'] = _dt
        for _sub in ['dir_util', 'file_util', 'errors', 'version', 'command']:
            try:
                sys.modules[f'distutils.{_sub}'] = getattr(_dt, _sub)
            except AttributeError:
                pass
        print('distutils patched')
    except Exception as _e:
        print(f'Could not patch distutils: {_e}')


CalledProcessError: Command '['C:\\Users\\docto\\miniconda3\\envs\\ultraseg\\python.exe', '-m', 'pip', 'install', '-q', 'nnunet==1.7.1']' returned non-zero exit status 1.

In [ ]:
import glob, os
import numpy as np
from musegai import api
from musegai.api import Volume

# ── Paths ────────────────────────────────────────────────────────────────────
WEIGHTS_DIR = os.path.expanduser('~/nnUNet_trained_models')
OUTPUT_DIR  = os.path.expanduser('~/museg_thigh_segs_dixon')
DATA_ROOT   = os.path.expanduser('~/myosegmenTUM')

# Iterate over FAT stacks; WATER counterparts are loaded alongside
IMAGE_GLOB = os.path.join(DATA_ROOT, '*', 'ImageData', '*_FAT', '*_FAT_stack*.nii')

LABEL_MAP = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

def fat_to_water_path(fat_path):
    return fat_path.replace('_FAT', '_WATER').replace('/_FAT/', '/_WATER/')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Data root exists:', os.path.isdir(DATA_ROOT))
print('Weights dir exists:', os.path.isdir(WEIGHTS_DIR))
fat_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(fat_files)} FAT stacks')
if fat_files:
    w = fat_to_water_path(fat_files[0])
    print(f'  FAT example  : {fat_files[0]}')
    print(f'  WATER example: {w}')
    print(f'  WATER exists : {os.path.exists(w)}')

In [ ]:
# Download the custom nnU-Net trainer from GitHub and register it
import urllib.request, importlib
from pathlib import Path

trainer_url  = ('https://raw.githubusercontent.com/fabianbalsiger/museg-ai'
                '/main/docker/nnUNetTrainerV2_MUSEGAI.py')
nnunet_trainers = (Path(importlib.util.find_spec('nnunet').origin).parent
                   / 'training' / 'network_training')
trainer_dest = nnunet_trainers / 'nnUNetTrainerV2_MUSEGAI.py'

if not trainer_dest.exists():
    urllib.request.urlretrieve(trainer_url, trainer_dest)
    print('Trainer downloaded to', trainer_dest)
else:
    print('Trainer already present')

os.environ['RESULTS_FOLDER']     = WEIGHTS_DIR
os.environ['nnUNet_results']      = WEIGHTS_DIR
os.environ['nnUNet_raw']          = '/tmp/nnunet_raw'
os.environ['nnUNet_preprocessed'] = '/tmp/nnunet_preprocessed'
print('nnU-Net env vars set')


In [ ]:
import os, shutil, torch, numpy as np, scipy.ndimage as _snd
from unittest.mock import MagicMock
import musegai.api as api_module

api_module.docker = MagicMock()

# Guard against double-patching on re-run
if not getattr(torch.load, '_patched', False):
    _orig_torch_load = torch.load
    def _patched_torch_load(f, *args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(f, *args, **kwargs)
    _patched_torch_load._patched = True
    torch.load = _patched_torch_load

_orig_map_coords = _snd.map_coordinates
def _safe_map_coordinates(input, coordinates, **kwargs):
    input = np.asarray(input, dtype=np.float64)
    return _orig_map_coords(input, coordinates, **kwargs)
_snd.map_coordinates = _safe_map_coordinates
from nnunet.preprocessing import preprocessing as _nnunet_pp
_nnunet_pp.map_coordinates = _safe_map_coordinates

def _run_model_native(model, indir, outdir):
    print('Running nnU-Net inference')
    model_folder = os.path.join(
        WEIGHTS_DIR, 'nnUNet', '3d_fullres', 'Task503_MuscleThigh',
        'nnUNetTrainerV2_MUSEGAI__nnUNetPlansv2.1'
    )
    print(f'  Model folder exists: {os.path.isdir(model_folder)}')
    from nnunet.inference.predict import predict_from_folder
    predict_from_folder(
        model=model_folder,
        input_folder=str(indir),
        output_folder=str(outdir),
        folds=None, save_npz=False,
        num_threads_preprocessing=2, num_threads_nifti_save=2,
        lowres_segmentations=None, part_id=0, num_parts=1,
        tta=False, mixed_precision=False, overwrite_existing=True,
        mode='normal', overwrite_all_in_gpu=True, step_size=0.5,
        checkpoint_name='model_final_checkpoint',
        segmentation_export_kwargs=None, disable_postprocessing=False,
    )
    shutil.copy('/home/ubuntu/museg-src/docker/labels_thigh.txt',
                os.path.join(str(outdir), 'labels.txt'))

api_module._run_model = _run_model_native
print('All patches applied — ready to run inference')

In [ ]:
for fat_path in fat_files:
    # Derive stem and output path from FAT filename
    fat_stem = os.path.splitext(os.path.basename(fat_path))[0]   # HV001_1_FAT_stack1
    subject  = fat_stem.split('_FAT')[0]                          # HV001_1
    import re; stack_n = re.search(r'stack(\d+)', fat_stem).group(1)
    stem     = f'{subject}_stack{stack_n}'                        # HV001_1_stack1
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_museg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    water_path = fat_to_water_path(fat_path)
    if not os.path.exists(water_path):
        print(f'WATER not found for {stem}, skipping')
        continue

    print(f'\nProcessing: {stem}')

    # Load FAT and WATER, then compute Dixon in-phase and out-of-phase
    fat_vol   = Volume.load(fat_path)
    water_vol = Volume.load(water_path)

    inphase_arr  = fat_vol.array.astype(float) + water_vol.array.astype(float)
    outphase_arr = fat_vol.array.astype(float) - water_vol.array.astype(float)

    inphase_vol  = Volume(inphase_arr,  **fat_vol.metadata)
    outphase_vol = Volume(outphase_arr, **fat_vol.metadata)

    print(f'  Shape: {fat_vol.shape}  Spacing: {fat_vol.spacing}')

    results, labels = api.segment_volumes(
        {stem: [inphase_vol, outphase_vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[stem]
    segmentation.save(out_path)
    print(f'  Saved -> {out_path}')

    seg_arr = segmentation.array
    print(f'  Labels present: {sorted(np.unique(seg_arr).tolist())}')
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    print(f'  {"-"*45}')
    for idx, name in LABEL_MAP.items():
        n = int((seg_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nAll done.')